In [35]:
import requests
import json
import zipfile
import io
import pandas as pd
from datetime import datetime, timedelta

with open('auth_response.json', 'r') as f:
    data = json.load(f)
    access_token = data['access_token']

with open('ercot_credentials.json', 'r') as f:
    credentials = json.load(f)
    SUBSCRIPTION_KEY = credentials['ERCOT_API_KEY']

PRODUCTS_URL = "https://api.ercot.com/api/public-reports"
HEADERS = {"Authorization": "Bearer " + access_token, "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY}




# ============ CONFIGURATION =============
API_BASE = "https://api.ercot.com/api/public-reports"

# EMIL product IDs
LOAD_EMIL = "NP6-346-CD"
PRICE_EMIL = "NP6-785-ER"
PARAMS = {
    "SCEDTimestampFrom": "2024-01-01T00:00:00",  # start time (ISO format)
    "SCEDTimestampTo": "2025-01-01T00:00:00"     # end time (ISO format)
}
# region / zone to extract
ZONE = "LZ_HOUSTON"  # or any Load Zone / Hub


In [36]:
def list_reports(emil_id):
    """List report artifacts for a given EMIL product."""
    resp = requests.get(f"{API_BASE}/{emil_id}", headers=HEADERS)
    resp.raise_for_status()
    jr = resp.json()
    # The artifacts are under "_embedded" → "reports" or similar
    return jr.get("_embedded", {}).get("artifacts", [])

list_reports(LOAD_EMIL)

[]

In [37]:
PRODUCTS_URL = f"https://api.ercot.com/api/public-reports/{LOAD_EMIL}/data"

product_response = requests.get(PRODUCTS_URL, headers=HEADERS)

print (product_response.text)

{ "statusCode": 404, "message": "Resource not found" }


In [38]:
load=requests.get("https://api.ercot.com/api/public-reports/np6-346-cd/act_sys_load_by_fzn", headers=HEADERS, params=PARAMS)
print(load.text)

{"timestamp":"2025-10-07 07:34:05","code":400,"status":"BAD_REQUEST","message":"One or more of the query parameters specified are not available for this resource.","data":["SCEDTimestampFrom","SCEDTimestampTo"]}


In [41]:
from datetime import datetime, timedelta

start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 3)
days = (end_date - start_date).days

for i in range(days):
    date_str = (start_date + timedelta(days=i)).strftime("%Y-%m-%d")
    params = {"deliveryDate": date_str}
    load = requests.get(
        "https://api.ercot.com/api/public-reports/np6-346-cd",
        headers=HEADERS,
        params=params
    )
    # Process each day’s data here
    print(load.text)


{"emilId":"NP6-346-CD","name":"Actual System Load by Forecast Zone","description":"A daily report of Actual System Load by Forecast Zone for each hour of the previous operating day.","status":"Active","reportTypeId":14836,"audience":"Public","generationFrequency":"Chron - Daily","securityClassification":"Public","lastUpdated":"2021-02-07","firstRun":"2017-06-29","eceii":null,"channel":"Public, EWS","userGuide":null,"postingType":"Report","market":"Nodal","extractSubscriber":null,"xsdName":"Current Day Reports XSD","misPostingLocation":null,"certificateRole":null,"fileType":"zip, csv, xml","ddlName":null,"misDisplayDuration":31,"archiveDuration":2555,"notificationType":null,"contentType":"DATA","downloadLimit":1000,"lastPostDatetime":"2025-10-07T05:50:00","bundle":1,"protocolRules":{"NP6.3.2(4)":"https://www.ercot.com/mp/data-products?protocolRules=NP6.3.2(4)"},"artifacts":[{"reportTypeId":14836,"displayName":"Actual System Load by Forecast Zone","_links":{"endpoint":{"href":"https://ap

In [43]:
load.json()

{'emilId': 'NP6-346-CD',
 'name': 'Actual System Load by Forecast Zone',
 'description': 'A daily report of Actual System Load by Forecast Zone for each hour of the previous operating day.',
 'status': 'Active',
 'reportTypeId': 14836,
 'audience': 'Public',
 'generationFrequency': 'Chron - Daily',
 'securityClassification': 'Public',
 'lastUpdated': '2021-02-07',
 'firstRun': '2017-06-29',
 'eceii': None,
 'channel': 'Public, EWS',
 'userGuide': None,
 'postingType': 'Report',
 'market': 'Nodal',
 'extractSubscriber': None,
 'xsdName': 'Current Day Reports XSD',
 'misPostingLocation': None,
 'certificateRole': None,
 'fileType': 'zip, csv, xml',
 'ddlName': None,
 'misDisplayDuration': 31,
 'archiveDuration': 2555,
 'notificationType': None,
 'contentType': 'DATA',
 'downloadLimit': 1000,
 'lastPostDatetime': '2025-10-07T05:50:00',
 'bundle': 1,
 'protocolRules': {'NP6.3.2(4)': 'https://www.ercot.com/mp/data-products?protocolRules=NP6.3.2(4)'},
 'artifacts': [{'reportTypeId': 14836,
 